In [3]:
%reload_ext autoreload
%autoreload 2
import sys

sys.path.append("../")
from epd_process_funcs import *

from visualisation import *

In [4]:
solar = pd.read_csv(
    "/home/hossein/CICCADA/BOM_NCI/2023/NCI_processed_grouped_01_02.csv"
)
solar["time"] = pd.to_datetime(solar["time"])
solar["time"] = solar["time"].dt.tz_localize("utc")
solar["time"] = solar["time"].dt.tz_convert(pytz.FixedOffset(9.5 * 60))
solar["postcode"] = solar["postcode"].astype(int)
solar.rename(
    columns={"surface_global_irradiance": "GHI", "direct_normal_irradiance": "DNI"},
    inplace=True,
)

In [5]:
edp_path = "4) Data/EDP SA 2023 Data"
edp_files = glob(edp_path + "/SA_site_edp_2023_S*.csv")
df = pd.concat(
    [pd.read_csv(i) for i in edp_files if os.path.getsize(i) > 0]
).reset_index(drop=True)

In [6]:
meta_data1 = pd.read_csv(edp_path + "/edp_sites_metadata_sa_postcode.csv")
meta_data2 = pd.read_csv(edp_path + "/edp_sites_metadata59239829.csv")
meta_data2 = meta_data2[meta_data2["state"] == "SA"]
meta_data3 = meta_data2.merge(
    meta_data1[["edp_site_id", "postcode"]], on="edp_site_id", how="left"
)
meta_data3["Srated"] = (
    meta_data3["inverter_ac_rating_kw"] * meta_data3["inverter_count"]
)
meta_data3["Srated"] = meta_data3.apply(
    lambda row: (
        row["inverter_ac_rating_kw"]
        if pd.isna(row["inverter_count"])
        else row["Srated"]
    ),
    axis=1,
)
meta_data2 = (
    meta_data3.groupby(["edp_site_id", "postcode"]).agg({"Srated": "sum"}).reset_index()
)

In [7]:
df5 = process_edp(df, meta_data2, 10.5 * 60)

In [8]:
df6 = df5.copy()
df6["DP"] = df6.groupby(["edp_site_id"])["active_power"].transform(lambda x: x.diff())
df6["DP_r"] = df6["DP"].clip(upper=0).round()

In [14]:
ii = 0

In [24]:
df8 = df6.query(
    f"DP_r <= -1 and active_power < .05 and  active_power >= -.05 and voltage_avg >= 258"
).reset_index(drop=True)
# df8 = df8[df8['time'].dt.hour < 17]
site_ids = df8["edp_site_id"].unique()[0]
time_t = df8.query(f"edp_site_id == '{site_ids}'")["time"].to_list()[ii]
# ii= ii + 1
t0 = time_t.replace(hour=0, minute=0, second=0, microsecond=0)
t1 = t0 + pd.Timedelta(days=1)
print(f"Time of tripping: {time_t}")
print(f"site_id: {site_ids}, postcode: {df8['postcode'].unique()[0]}")
df8 = df6.copy()
df8 = df8[df8["edp_site_id"] == site_ids].reset_index(drop=True)
start_time = t0  # In sydney local time
end_time = t1  # In sydney local time

num_ticks = 73
save_as = ""
x_label = "time"
y_labels = ["Active power (kW)", "voltage (V)", "GHI", "Cloud type"]
plt_config = {
    "active_power": [0, 0, "-", None, None],
    "voltage_avg": [0, 1, "-", None, None],
}
# y_labels = ['Active power (kW)',  'Apparent power (kVA)']
# plt_config = {'active_power': [0, 0, '-', -2.5, 4.5], 'apparent_power': [1, 1, '-', None, None]}
color_nights = False
# color_by = 'group'
color_by = "attribute"
ax_digit = "1.2f"
a = my_plot4(
    start_time,
    end_time,
    df8,
    plt_config=plt_config,
    ax_digit=ax_digit,
    group_attr="edp_site_id",
    time_attr="time",
    color_nights=color_nights,
    cmap="viridis",
    figsize=[17 / 2.54, 2.0],
    same_scale=1,
    fontsize=7,
    fontname="Times New Roman",
    plot_total=False,
    plot_total_func=["sum", [lambda x: max(x), "max"]],
    num_ticks=num_ticks,
    num_yticks=10,
    dpi=300,
    x_format="%H:%M",
    legend_loc=["lower left", "upper right", "lower left", "upper right"],
    x_label=x_label,
    y_labels=y_labels,
    color_by=color_by,
    plot_period=np.timedelta64(1, "D"),
    save_as=save_as,
    rotation=60,
    step=0,
    gridwidth=[0.5, 0.5],
    legend_join="-",
    title="",
    legend_i=0,
    title_i=0,
    only1title=1,
    onlyntime=0,
)
a.do()

IndexError: list index out of range

In [46]:
df8 = df6.query(
    f"DP_r < -1 and active_power < .1 and  active_power >= -.05 and voltage_avg > 250 and time < '2023-02-28 00:00:00+1030'"
).reset_index(drop=True)
site_ids = df8["edp_site_id"].unique()[0]
time_t = df8.query(f"edp_site_id == '{site_ids}'")["time"].to_list()[0]
t0 = time_t.replace(hour=0, minute=0, second=0, microsecond=0)
t1 = t0 + pd.Timedelta(days=1)
print(f"Time of tripping: {time_t}")
print(f"site_id: {site_ids}, postcode: {df8['postcode'].unique()[0]}")
df8 = df6.copy()
df8 = df8[df8["edp_site_id"] == site_ids].reset_index(drop=True)
df8 = df8.merge(solar, on=["time", "postcode"], how="left")
df8 = (
    df8.sort_values(["postcode", "time"])
    .groupby("postcode")
    .apply(
        lambda group: (
            group.set_index("time").interpolate(method="time", limit=1).reset_index()
        )
    )
    .reset_index(drop=True)
)
start_time = t0  # In sydney local time
end_time = t1  # In sydney local time

num_ticks = 73
save_as = ""
x_label = "time"
y_labels = ["Active power (kW)", "voltage (V)", "GHI", "Cloud type"]
plt_config = {
    "active_power": [1, 0, "-", None, None],
    "voltage_avg": [1, 1, "-", None, None],
    "GHI": [0, 0, "-", None, None],
    "cloud_type": [0, 1, "-", None, None],
}
# y_labels = ['Active power (kW)',  'Apparent power (kVA)']
# plt_config = {'active_power': [0, 0, '-', -2.5, 4.5], 'apparent_power': [1, 1, '-', None, None]}
color_nights = False
# color_by = 'group'
color_by = "attribute"
ax_digit = "1.2f"
a = my_plot4(
    start_time,
    end_time,
    df8,
    plt_config=plt_config,
    ax_digit=ax_digit,
    group_attr="edp_site_id",
    time_attr="time",
    color_nights=color_nights,
    cmap="viridis",
    figsize=[17 / 2.54, 2.0],
    same_scale=1,
    fontsize=7,
    fontname="Times New Roman",
    plot_total=False,
    plot_total_func=["sum", [lambda x: max(x), "max"]],
    num_ticks=num_ticks,
    num_yticks=10,
    dpi=300,
    x_format="%H:%M",
    legend_loc=["lower left", "upper right", "lower left", "upper right"],
    x_label=x_label,
    y_labels=y_labels,
    color_by=color_by,
    plot_period=np.timedelta64(1, "D"),
    save_as=save_as,
    rotation=60,
    step=0,
    gridwidth=[0.5, 0.5],
    legend_join="-",
    title="",
    legend_i=0,
    title_i=0,
    only1title=1,
    onlyntime=0,
)
a.do()

IndexError: index 0 is out of bounds for axis 0 with size 0